In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import xgboost as xgb

df = pd.read_csv('/content/ats_resume_dataset_elite_v3.csv')

# Feature Engineering
edu_map = {'Bachelors': 0, 'Masters': 1, 'PhD': 2}
df['edu_encoded'] = df['education_level'].map(edu_map).fillna(0)

def skill_overlap(row):
    resume_skills = set(str(row['resume_skills']).lower().split(', '))
    required_skills = set(str(row['required_skills']).lower().split(', '))
    if not required_skills: return 0
    return len(resume_skills & required_skills) / len(required_skills)

df['computed_skill_overlap'] = df.apply(skill_overlap, axis=1)
df['exp_gap'] = df['experience_years'] - df['job_experience_required']
df['exp_gap_clipped'] = df['exp_gap'].clip(-5, 10)

# Text feature: TF-IDF on resume
tfidf = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1,2))
tfidf_features = tfidf.fit_transform(df['resume_text'].fillna('')).toarray()
tfidf_df = pd.DataFrame(tfidf_features, columns=[f'tfidf_{i}' for i in range(tfidf_features.shape[1])])

# Structured features
structured = df[['skill_match_score', 'experience_match', 'education_match',
                  'final_score', 'similarity_score', 'edu_encoded',
                  'computed_skill_overlap', 'exp_gap_clipped', 'experience_years',
                  'job_experience_required']].fillna(0)

X = np.hstack([structured.values, tfidf_features])
y = df['shortlisted'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Model 1: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Model 2: XGBoost
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                random_state=42, eval_metric='logloss', verbosity=0)
xgb_model.fit(X_train, y_train)

# Model 3: Deep Neural Network (MLP)
mlp = MLPClassifier(hidden_layer_sizes=(256, 128, 64, 32), activation='relu',
                    max_iter=300, random_state=42, early_stopping=True,
                    validation_fraction=0.1, learning_rate_init=0.001)
mlp.fit(X_train, y_train)

# Ensemble predictions
rf_pred = rf.predict(X_test)
xgb_pred = xgb_model.predict(X_test)
mlp_pred = mlp.predict(X_test)

# Soft voting ensemble
rf_prob = rf.predict_proba(X_test)[:,1]
xgb_prob = xgb_model.predict_proba(X_test)[:,1]
mlp_prob = mlp.predict_proba(X_test)[:,1]
ensemble_prob = (rf_prob * 0.35 + xgb_prob * 0.45 + mlp_prob * 0.20)
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

print("=== MODEL ACCURACIES ===")
print(f"Random Forest: {accuracy_score(y_test, rf_pred):.4f}")
print(f"XGBoost:       {accuracy_score(y_test, xgb_pred):.4f}")
print(f"MLP (DL):      {accuracy_score(y_test, mlp_pred):.4f}")
print(f"Ensemble:      {accuracy_score(y_test, ensemble_pred):.4f}")

# Save everything
bundle = {
    'rf': rf,
    'xgb': xgb_model,
    'mlp': mlp,
    'tfidf': tfidf,
    'scaler': scaler,
    'edu_map': edu_map,
    'structured_cols': structured.columns.tolist(),
    'n_tfidf': tfidf_features.shape[1]
}
with open('/content/model_bundle.pkl', 'wb') as f:
    pickle.dump(bundle, f)

# Also save job roles and their required skills from dataset
job_info = df.groupby('job_role').agg({
    'required_skills': 'first',
    'job_experience_required': 'mean',
    'job_description': 'first'
}).reset_index().to_dict('records')
with open('/content/job_info.json', 'w') as f:
    json.dump(job_info, f)

print("\nModels saved!")
print(f"Classification Report (Ensemble):")
print(classification_report(y_test, ensemble_pred))

=== MODEL ACCURACIES ===
Random Forest: 0.9983
XGBoost:       0.9992
MLP (DL):      0.9650
Ensemble:      0.9992

Models saved!
Classification Report (Ensemble):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       289
           1       1.00      1.00      1.00       911

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200

